In [ ]:
%pip install --no-cache-dir \
  "transformers==5.10.2" \
  "tokenizers==0.22.2" \
  "safetensors==0.8.0" \
  "huggingface-hub==1.18.0" \
  "datasets==4.5.0" \
  "accelerate==1.10.1" \
  "trl==0.29.0" \
  "peft==0.19.1" \
  "sentencepiece==0.2.1" \
  "protobuf==6.33.2" \
  "soxr==1.1.0" \
  "Pillow==12.2.0"

# 1. 데이터 전처리

In [ ]:
import torch

print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("has float8_e8m0fnu:", hasattr(torch, "float8_e8m0fnu"))

if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset, Dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import LoraConfig, AutoPeftModelForCausalLM
from trl import SFTConfig, SFTTrainer

In [ ]:
# 1. 민간 민원 상담 SFT 데이터셋 로드
# train/validation 파일이 이미 나뉘어 있으므로 노트북에서 다시 분할하지 않음
dataset = load_dataset(
    "json",
    data_files={
        "train": "data/train.jsonl",
        "validation": "data/validation.jsonl",
    },
)


# 2. 로드된 split별 데이터 크기 확인
print("Train 원본 데이터 크기:", len(dataset["train"]))
print("Validation 원본 데이터 크기:", len(dataset["validation"]))

# 3. OpenAI messages format으로 데이터 변환을 위한 함수
def format_data(sample):
    return {
        "messages": [
            {
                "role": "system",
                "content": sample["system_prompt"],
            },
            {
                "role": "user",
                "content": sample["user_prompt"],
            },
            {
                "role": "assistant",
                "content": str(sample["assistant"])
            },
        ],
    }

# 4. 이미 분리된 train/validation split을 messages format으로 변환
train_dataset = [format_data(sample) for sample in dataset["train"]]
test_dataset = [format_data(sample) for sample in dataset["validation"]]

# 5. 최종 데이터셋 크기 출력
print(f"데이터셋 변환 결과: Train {len(train_dataset)}개, Validation {len(test_dataset)}개")

In [ ]:
train_dataset[0]["messages"]

## Dataset 객체로 변환하는 이유

- 앞 Cell에서는 각 샘플을 `messages` 형식의 리스트로 변환합니다.
- 리스트 상태에서도 샘플 확인은 가능하지만, 이후 `SFTTrainer` 입력에는 Hugging Face `Dataset` 객체가 더 적합합니다.
- 따라서 `Dataset.from_list()`로 다시 감싸서 학습 Cell에서 안정적으로 사용할 수 있게 합니다.

In [ ]:
# 리스트 형태를 Hugging Face Dataset 객체로 변경
# 이후 SFTTrainer 입력과 Dataset 메서드 사용을 위해 변환
print("변환 전 train_dataset:", type(train_dataset))
print("변환 전 test_dataset:", type(test_dataset))
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)
print("변환 후 train_dataset:", type(train_dataset))
print("변환 후 test_dataset:", type(test_dataset))

In [ ]:
train_dataset[0]

# 2. 모델 로드 및 템플릿 적용

- Instruction Tuned(google/gemma-4-E4B-it)
- 사용자의 지시를 따르는 대화/명령 수행용으로 추가 학습된 모델

| 구분 | Base 모델 | Instruction-tuned 모델 |
|---|---|---|
| 모델명 예시 | `gemma-4-E4B` | `gemma-4-E4B-it` |
| 학습 목적 | 다음 토큰을 자연스럽게 예측 | 사용자의 지시를 읽고 적절히 응답 |
| 입력 반응 | 입력 문장을 이어 쓰는 경향이 강함 | 질문, 명령, 역할 지시에 맞춰 답변 |
| 대화 형식 | chat template이 없거나 약할 수 있음 | system/user/assistant 구조에 맞춰 학습됨 |
| SFT 시작점 | 지시 수행 방식까지 학습시켜야 함 | 이미 지시 수행 능력이 있어 작업 형식 조정에 유리 |
| 필요한 데이터 | 상대적으로 더 많을 수 있음 | 상대적으로 적은 데이터로도 목적 행동을 맞추기 쉬움 |
| 주요 사용처 | continued pretraining, 직접 instruction tuning | 챗봇, Agent, instruction SFT, 서비스 응답 |

In [ ]:
# 허깅페이스 모델 ID
model_id = "google/gemma-4-E4B-it"

# 모델과 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# 템플릿 적용
text = tokenizer.apply_chat_template(
    train_dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False
)
print(text)

## 모델 로드 메모

- `model_id`: `google/gemma-4-E4B-it`
- `it`: Instruction Tuned, 지시 응답용
- `AutoModelForCausalLM`: assistant 응답 생성용 causal LM 로드
- `device_map="auto"`: 사용 가능한 GPU에 자동 배치
- `dtype=torch.bfloat16`: VRAM 절약, L40S 지원
- `AutoTokenizer`: 모델 전용 토크나이저 로드
- `apply_chat_template`: messages를 Gemma 채팅 템플릿으로 변환
- RunPod L40S: 48GB VRAM, 4B급 LoRA/SFT 실습에 충분
- A100: 대역폭이 커서 대규모 학습에 더 유리
- OOM 대응 순서: batch size 축소 -> gradient accumulation 조정 -> max sequence length 축소

# 3. LoRA와 SFTConfig 설정

In [ ]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=8,
    bias="none",
    target_modules=r".*language_model\..*\.(q_proj|v_proj)",
    task_type="CAUSAL_LM",
)

## LoRA 설정 메모

- LoRA: Low-Rank Adaptation, 전체 모델 대신 일부 저랭크 어댑터만 학습
- `r=8`: 표현력 조절, 작으면 가볍고 크면 복잡
- `lora_alpha=32`: LoRA 변화량 스케일, 보통 `alpha / r` 비율로 영향 조절
- `lora_dropout=0.1`: 과적합 방지, 학습 중 일부 연결 비활성화
- `bias="none"`: bias 학습 제외, 메모리 절약
- `target_modules`: LoRA 적용 레이어 선택
- 현재 대상: `q_proj`, `v_proj` 중심 Attention 레이어
- `task_type="CAUSAL_LM"`: 다음 토큰 예측 기반 생성 태스크

In [ ]:
# 최대 길이
max_seq_length=16384

In [ ]:
args = SFTConfig(
    output_dir="gemma4-e4b-civil-complaint-sft",  # 저장 디렉토리와 저장소 ID
    num_train_epochs=3,                           # 전체 데이터셋 학습 반복 횟수
    per_device_train_batch_size=4,                # GPU당 학습 배치 크기
    per_device_eval_batch_size=4,                 # GPU당 평가 배치 크기
    gradient_accumulation_steps=12,               # effective batch = 4 x 12 = 48
    gradient_checkpointing=True,                  # VRAM 절약
    optim="adamw_torch_fused",                   # Fused AdamW
    logging_steps=10,                             # 로그 기록 주기
    save_strategy="steps",                       # step 기준 저장
    save_steps=50,                                # 체크포인트 저장 주기
    eval_strategy="steps",                       # step 기준 평가
    eval_steps=50,                                # 평가 주기
    save_total_limit=2,                           # 최근 체크포인트 2개 유지
    load_best_model_at_end=True,                  # 종료 시 최적 모델 로드
    metric_for_best_model="eval_loss",           # 최적 모델 기준
    greater_is_better=False,                      # eval_loss는 낮을수록 좋음
    bf16=True,                                    # bfloat16 사용
    learning_rate=1e-4,                           # LoRA 초기 학습률
    lr_scheduler_type="cosine",                  # cosine 스케줄러
    warmup_ratio=0.1,                             # 전체 스텝의 10% 워밍업
    weight_decay=0.01,                            # L2 정규화
    max_grad_norm=1.0,                            # 그래디언트 클리핑
    dataloader_num_workers=4,                     # 데이터 로딩 워커 수
    dataloader_pin_memory=True,                   # GPU 전송 최적화
    push_to_hub=False,                            # 허브 업로드 안 함
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to=[],
    max_length=max_seq_length,
)

## SFTConfig 설정 메모

### TrainingArguments(SFTConfig)

- `TrainingArguments`: Hugging Face Trainer 공통 학습 설정
- `SFTConfig`: TRL `SFTTrainer`용 설정, `TrainingArguments` 계열 옵션 + SFT 전용 옵션
- 공통 예: batch size, epoch, learning rate, scheduler, checkpoint, dataloader
- SFT 전용 예: `max_length`, `dataset_kwargs`, `packing`, `assistant_only_loss`

### 배치 및 학습 설정

샘플 4개씩 계산하고, 12번 누적한 뒤 샘플 48개 단위로 가중치를 한 번 업데이트합니다.

- `per_device_train_batch_size=4`: GPU당 학습 배치 크기
- `per_device_eval_batch_size=4`: GPU당 평가 배치 크기
- `gradient_accumulation_steps=12`: 그래디언트 누적, effective batch = 4 x 12 = 48
- `num_train_epochs=3`: 전체 데이터셋 학습 반복 횟수

### 학습률 설정

- `learning_rate=1e-4`: 초기 학습률, LoRA에서 흔히 쓰는 범위
- `lr_scheduler_type="cosine"`: cosine 스케줄러, 부드러운 감소
- `warmup_ratio=0.1`: 전체 스텝의 10% 워밍업

### 정규화 및 안정성

- `weight_decay=0.01`: L2 정규화, 과적합 완화
- `max_grad_norm=1.0`: 그래디언트 클리핑, 학습 안정성

### 메모리 및 성능 최적화

- `gradient_checkpointing=True`: VRAM 절약, 속도는 느려질 수 있음
- `bf16=True`: bfloat16 혼합 정밀도 학습
- `optim="adamw_torch_fused"`: Fused AdamW, 최적화 속도 개선

### 체크포인트 및 평가

RunPod 작업은 중단될 수 있으므로 체크포인트 저장 주기와 보관 개수가 중요합니다.

- `save_strategy="steps"`: step 기준 저장
- `save_steps=50`: 50스텝마다 체크포인트 저장
- `eval_strategy="steps"`: step 기준 평가
- `eval_steps=50`: 50스텝마다 validation 평가
- `save_total_limit=2`: 최근 체크포인트 2개만 유지
- `load_best_model_at_end=True`: 종료 시 최적 모델 로드
- `metric_for_best_model="eval_loss"`: 최적 모델 기준
- `greater_is_better=False`: eval_loss는 낮을수록 좋음

### 데이터 로딩

- `dataloader_num_workers=4`: 데이터 로딩 병렬 워커 수
- `dataloader_pin_memory=True`: GPU 전송 최적화
- `remove_unused_columns=False`: `messages` 컬럼 유지
- `dataset_kwargs={"skip_prepare_dataset": True}`: 직접 만든 `collate_fn` 사용
- `max_length=max_seq_length`: 최대 시퀀스 길이

# 4. 학습 중 전처리 함수: collate_fn

- Data Collator: 샘플 리스트를 모델 입력 배치로 변환
- 역할: chat template 적용, 토큰화, assistant 응답만 label로 남기기, padding, tensor 변환

In [ ]:
def collate_fn(batch):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    for example in batch:
        messages = example["messages"]

        # 전체 대화 토큰화
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        ).strip()
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=max_seq_length,
            padding=False,
            return_tensors=None,
        )

        # assistant 응답 시작 위치 계산
        prompt_text = tokenizer.apply_chat_template(
            messages[:-1],
            tokenize=False,
            add_generation_prompt=True,
        ).strip()
        prompt_tokenized = tokenizer(
            prompt_text,
            truncation=True,
            max_length=max_seq_length,
            padding=False,
            return_tensors=None,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        assistant_start = len(prompt_tokenized["input_ids"])

        # assistant 응답만 학습 대상
        labels = [-100] * assistant_start + input_ids[assistant_start:]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    # 배치 내 최대 길이에 맞춰 padding
    batch_max_length = max(len(input_ids) for input_ids in input_ids_list)

    for input_ids, attention_mask, labels in zip(input_ids_list, attention_mask_list, labels_list):
        padding_length = batch_max_length - len(input_ids)
        input_ids.extend([pad_token_id] * padding_length)
        attention_mask.extend([0] * padding_length)
        labels.extend([-100] * padding_length)

    # Trainer 입력 tensor 반환
    return {
        "input_ids": torch.tensor(input_ids_list, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask_list, dtype=torch.long),
        "labels": torch.tensor(labels_list, dtype=torch.long),
    }

In [ ]:
# collate_fn 테스트
example = train_dataset[0]
batch = collate_fn([example])
label_ids = [token_id for token_id in batch["labels"][0].tolist() if token_id != -100]
decoded_labels = tokenizer.decode(
    label_ids,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

print("처리된 배치 데이터")
print("- input_ids shape:", batch["input_ids"].shape)
print("- attention_mask shape:", batch["attention_mask"].shape)
print("- labels shape:", batch["labels"].shape)

print("\ninput_ids")
print(batch["input_ids"][0].tolist())

print("\nlabels")
print(batch["labels"][0].tolist())

print("\nlabels 디코딩 결과 (-100 제외)")
print(decoded_labels)

## input_ids와 labels는 어떻게 생성되는가?

### 목적

1. 모델 입력 전체: `input_ids`로 전달
2. 실제 학습 대상: assistant 응답만 유지
3. system, user, 응답 시작 태그: `-100` 마스킹으로 loss 제외

### 현상

1. 데이터셋 응답 role: `assistant`
2. Gemma 4 chat template 적용 후: assistant 응답 턴이 `model` 턴으로 렌더링
3. 전체 텍스트: 토크나이저를 거쳐 정수 토큰 시퀀스로 변환

예시: 민간 민원 상담 분류 데이터. 사용자는 상담 내용과 분류 지시 제공, 모델은 정답 label만 간결하게 응답.

```text
<|turn>system
너는 민간 민원 상담 데이터를 처리하는 상담 분석 AI다.<turn|>
<|turn>user
[작업] 분류
[세부 유형] 상담 내용
[지시] 이 상담은 일반 문의 상담, 업무 처리 상담, 고충 상담 중 어떤 유형인가?

[상담 내용]
고객: 예약 가능한 객실과 추가 요금을 알고 싶어요.
상담사: 예약 가능 여부와 요금을 확인해 드리겠습니다.<turn|>
<|turn>model
일반 문의 상담<turn|>
```

```python
input_ids = [
    # system 턴
    1001, 1002, 13, 1003, 1004, 1005, 1006, 1007, 13,
    # user 턴
    2001, 2002, 13, 2003, 2004, 2005, 2006, 2007, 2008, 13,
    # model 응답 시작 태그
    3001, 3002, 13,
    # 일반 문의 상담<turn|>
    5001, 5002, 5003, 5004
]
```

### 문제

1. `input_ids` 전체를 label로 사용 시: system/user 입력까지 정답처럼 학습
2. SFT 목적: 질문 암기가 아니라 assistant 응답 형식 학습
3. assistant 응답 이전 토큰: 학습 손실에서 제외 필요

```python
labels = [
    # system 부분 마스킹
    -100, -100, -100, -100, -100, -100, -100, -100, -100,
    # user 부분 마스킹
    -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
    # model 응답 시작 태그 마스킹
    -100, -100, -100,
    # 일반 문의 상담<turn|> (마스킹 없음)
    5001, 5002, 5003, 5004
]
```

### 처리 순서

1. `messages[:-1]` + `add_generation_prompt=True` 적용
2. 응답 시작 직전 토큰 길이: `assistant_start`
3. `assistant_start` 이전: `-100` 마스킹
4. `assistant_start` 이후: assistant 응답 토큰만 유지

```python
prompt_text = tokenizer.apply_chat_template(
    messages[:-1],
    tokenize=False,
    add_generation_prompt=True,
).strip()

prompt_tokenized = tokenizer(
    prompt_text,
    truncation=True,
    max_length=max_seq_length,
    padding=False,
    return_tensors=None,
)

assistant_start = len(prompt_tokenized["input_ids"])
```

```python
labels = [-100] * assistant_start + input_ids[assistant_start:]
```

결과: system/user/응답 시작 태그는 `-100`, 실제 model 응답 내용만 학습 대상.

# 5. 학습

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
)

In [ ]:
# 학습 시작
trainer.train()   # 모델이 자동으로 허브와 output_dir에 저장됨

# 모델 저장
trainer.save_model()   # 최종 모델을 저장

# 6. 테스트 데이터 준비

In [ ]:
prompt_lst = []
label_lst = []

for prompt in test_dataset["messages"]:
   input = tokenizer.apply_chat_template(
       prompt[:-1],
       tokenize=False,
       add_generation_prompt=True
   )
   label = prompt[-1]["content"]
   prompt_lst.append(input)
   label_lst.append(label)

In [ ]:
print(prompt_lst[0])

In [ ]:
print(label_lst[0])

# 7. 파인 튜닝 모델 테스트

## 파인튜닝 모델 테스트 메모

학습이 끝난 LoRA Adapter를 다시 로드하고, validation prompt에 대한 응답을 생성해 정답 label과 비교합니다.

- `AutoPeftModelForCausalLM`: base LLM + LoRA Adapter 로드
- `peft_model_id`: LoRA Adapter 저장 경로
- 현재 경로: `gemma4-e4b-civil-complaint-sft`
- 최종 모델 테스트: 학습 Cell의 `output_dir` 사용
- 중간 체크포인트 테스트: `checkpoint-*` 경로로 변경
- `fine_tuned_model`: LoRA Adapter가 결합된 테스트용 모델
- `device_map="auto"`: 사용 가능한 GPU에 자동 배치
- `pipeline("text-generation")`: prompt 입력 후 모델 응답 생성
- 비교 대상: validation 정답 label vs fine-tuned 모델 응답

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline

In [ ]:
peft_model_id = "gemma4-e4b-civil-complaint-sft"

fine_tuned_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_id,
    device_map="auto",
    dtype=torch.bfloat16
)

pipe = pipeline("text-generation", model=fine_tuned_model, tokenizer=tokenizer)

In [ ]:
eos_token = tokenizer("<turn|>", add_special_tokens=False)["input_ids"][0]

In [ ]:
eos_token

In [ ]:
def test_inference(pipe, prompt):
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    return outputs[0]['generated_text'][len(prompt):].strip()

임의로 테스트 데이터 10번부터 14번까지 확인해봅시다.


In [ ]:
def normalize_answer(text):
    return (
        text.replace("<turn|>", "")
        .replace("<end_of_turn>", "")
        .strip()
    )

for prompt, label in zip(prompt_lst[10:15], label_lst[10:15]):
    response = test_inference(pipe, prompt)

    pred = normalize_answer(response)
    gold = normalize_answer(label)

    print(f"    response:\n{response}")
    print(f"    label:\n{label}")
    print(f"    correct: {pred == gold}")
    print("-" * 50)

# 8. 기본 모델 테스트

이번에는 LoRA Adapter를 merge하지 않은 기본 모델로 테스트 데이터에 대해서 인퍼런스해보겠습니다.

In [ ]:
# fine-tuned 모델과 학습 객체 메모리 정리
import gc

for name in ["pipe", "fine_tuned_model", "trainer", "model"]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
base_model_id = "google/gemma-4-E4B-it"

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    dtype=torch.bfloat16
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
for prompt, label in zip(prompt_lst[10:15], label_lst[10:15]):
    # print(f"    prompt:\n{prompt}")
    print(f"    response:\n{test_inference(pipe, prompt)}")
    print(f"    label:\n{label}")
    print("-"*50)